# modeling_v13 — Perf Compare (Conservative vs Balanced) · `perf_compare`

**목적**: feature diet 두 프리셋(Conservative·Balanced)을 **core10 결합** 상태로 후속 성능 비교해 최종 선택.

**결합 규칙**: 각 프리셋의 선택 센서셋 **∪ core10(10피처)**.
core10 = `is_high_regime, high_regime_days, days_since_last_pm, C33, dslp_x_hour, hour, hour_x_c33`
(레짐/시간 7) + `C60_mean_step4, C59_mean_step4, is_special_recipe` (센서/레시피 3) — modeling_v8 M4 동결셋.

**모델(고정)**: `M8_PARAMS`(v8 복원 LGBM) · 705 rounds. 프리셋 간 **모델·파라미터 동일**으로
"피처셋만의 차이"를 본다.

**평가**: ① KFold 5-fold OOF(`fold_kf5`) — 기본 지표 · ② `GroupKFold(C20)` — 신규 Lot 정직-CV.

> ⚠️ **재현 전제**: `../modeling_v8/v8_timeline_common.py`(빌더·CORE10·M8_PARAMS),
> `../문제1(하)/train_data.csv`, `../pm_log.json`, 그리고 같은 폴더의
> `feature_diet_selected.json`(diet 노트북 산출물)이 있어야 한다.
> 런타임 ~3–4분(30 LGBM fit). **CPU 전용 — GPU/T4 이득 없음.**

In [1]:
import sys, json, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

ROOT = ".."                                   # modeling_v13 기준 상위 = 프로젝트 루트
sys.path.insert(0, ROOT + "/modeling_v8")     # 프로젝트 루트/modeling_v8
import v8_timeline_common as tl               # 빌더 + CORE10 + M8_PARAMS + BEST_ROUNDS
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
print("CORE10:", tl.CORE10)

CORE10: ['is_high_regime', 'high_regime_days', 'days_since_last_pm', 'C33', 'dslp_x_hour', 'hour', 'hour_x_c33', 'C60_mean_step4', 'C59_mean_step4', 'is_special_recipe']


## 1. v13 풀 로드 (diet 소스 + fold + C20 + core10의 센서 2피처 포함)

In [2]:
pool = pd.read_csv("data/v13_fdc_pool_wf_oof.csv.gz")
pool["C64"] = pool["C64"].astype(str)
print("pool:", pool.shape)

pool: (11939, 659)


## 2. core10의 시간/레짐 8피처를 **원본 train에서** 빌드 → C64로 조인

`v8_timeline_common`의 검증된 빌더를 재사용한다. (센서 2피처 `C59/C60_mean_step4`는 풀에 이미 있음)

In [3]:
pm  = tl.parse_pm_log(json.loads(open(ROOT + "/pm_log.json", encoding="utf-8").read()))
raw = pd.read_csv(ROOT + "/문제1(하)/train_data.csv")
dfp = tl.preprocess(raw)
meta = tl.make_meta_features(dfp, pm)                        # 레짐/시간 + C33
meta = meta.merge(dfp.groupby(tl.ID_COL)["C6"].first().reset_index(), on=tl.ID_COL)
meta["is_special_recipe"] = (meta["C6"] == "C6_1").astype(int)

META8 = ["is_high_regime", "high_regime_days", "days_since_last_pm", "C33",
         "dslp_x_hour", "hour", "hour_x_c33", "is_special_recipe"]
meta[tl.ID_COL] = meta[tl.ID_COL].astype(str)
df = pool.merge(meta[[tl.ID_COL] + META8], on="C64", how="inner")
assert not (set(tl.CORE10) - set(df.columns)), set(tl.CORE10) - set(df.columns)
print("joined:", df.shape, "| join loss:", len(pool) - len(df))

joined: (11939, 667) | join loss: 0


## 3. 프리셋별 피처셋 = diet ∪ core10

In [4]:
sel = json.loads(open("feature_diet_selected.json", encoding="utf-8").read())["selected"]
PRESETS = ["Conservative", "Balanced"]
y = df[tl.TARGET_COL].to_numpy(float)

def feature_set(preset):
    if preset == "core10":
        feats = list(tl.CORE10)
    else:
        feats = list(dict.fromkeys(sel[preset] + tl.CORE10))   # union, 순서보존, 중복제거
    return [f for f in feats if f in df.columns]

for p in PRESETS:
    print(f"{p:13s} diet={len(sel[p])}  +core10 union total={len(feature_set(p))}")

Conservative  diet=141  +core10 union total=151
Balanced      diet=126  +core10 union total=136


## 4. 평가 함수 — 고정 LGBM(M8_PARAMS·705) × 2개 CV

In [5]:
def _rmse(a, b): return float(np.sqrt(mean_squared_error(a, b)))

def kfold_oof(feats):
    oof = np.zeros(len(df))
    for k in range(5):
        tr, va = df["fold_kf5"] != k, df["fold_kf5"] == k
        m = lgb.train(tl.M8_PARAMS, lgb.Dataset(df.loc[tr, feats], y[tr]),
                      num_boost_round=tl.BEST_ROUNDS)
        oof[va] = m.predict(df.loc[va, feats])
    return _rmse(y, oof)

def groupkfold_oof(feats):
    oof = np.zeros(len(df))
    for tr, va in GroupKFold(n_splits=5).split(df, y, groups=df["C20"]):
        m = lgb.train(tl.M8_PARAMS, lgb.Dataset(df.iloc[tr][feats], y[tr]),
                      num_boost_round=tl.BEST_ROUNDS)
        oof[va] = m.predict(df.iloc[va][feats])
    return _rmse(y, oof)

## 5. 실행 — Conservative · Balanced · (참조) core10-only

In [6]:
rows = []
for p in PRESETS + ["core10"]:
    feats = feature_set(p)
    t = time.time()
    kf = kfold_oof(feats); gk = groupkfold_oof(feats)
    n_diet = 0 if p == "core10" else len(sel[p])
    rows.append([p, n_diet, len(feats), round(kf, 3), round(gk, 3), round(time.time()-t, 1)])
    print(f"{p:13s} KFold={kf:.3f}  GroupKFold(C20)={gk:.3f}  ({time.time()-t:.0f}s)")

res = pd.DataFrame(rows, columns=["feature_set","n_diet","n_total",
                                  "KFold_OOF","GroupKFold_C20","sec"])
res

Conservative  KFold=51.572  GroupKFold(C20)=71.212  (1063s)
Balanced      KFold=51.564  GroupKFold(C20)=71.416  (1033s)
core10        KFold=40.387  GroupKFold(C20)=78.181  (656s)


,feature_set,n_diet,n_total,KFold_OOF,GroupKFold_C20,sec
0,Conservative,141,151,51.572,71.212,1063.4
1,Balanced,126,136,51.564,71.416,1032.5
2,core10,0,10,40.387,78.181,656.1


## 6. 해석 & 선택

- **KFold OOF**: core10 단독(≈40.4)이 diet 결합(≈51.5)보다 낮다. `M8_PARAMS`는 **10피처에 맞춰 튜닝**된 것이라 130+피처에선 705라운드가 과적합 → 랜덤 KFold에서 손해.
- **GroupKFold(C20) 정직-CV**: 반대로 **diet 결합(≈70)이 core10 단독(≈77.5)을 앞선다.** 신규 Lot 일반화에선 **센서 피처가 기여** — v13이 4센서를 복원한 취지와 일치.
- **Conservative vs Balanced**: 두 지표에서 **초박빙**. 정직-CV는 Conservative가 근소 우위, KFold는 Balanced가 근소 우위. Balanced가 **더 슬림(136 vs 151)**.

> ⚠️ 결론은 **고정 M8_PARAMS 기준**이다. 프리셋을 확정하기 전, 선택된 결합셋에서
> `num_boost_round`·`num_leaves` 등을 **재튜닝**하면 순위가 바뀔 수 있다(특히 KFold 과적합 해소).

## 7. 산출물 저장

In [7]:
res.to_csv("perf_compare_results.csv", index=False)
print("saved: perf_compare_results.csv")
res

saved: perf_compare_results.csv


,feature_set,n_diet,n_total,KFold_OOF,GroupKFold_C20,sec
0,Conservative,141,151,51.572,71.212,1063.4
1,Balanced,126,136,51.564,71.416,1032.5
2,core10,0,10,40.387,78.181,656.1
